IMPORTING LIBRARIES

In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IMPORTING DATASET

In [36]:
dataset = pd.read_csv('rating.csv', sep=',')
print(dataset.head())

   userId  movieId  rating            timestamp
0       1      2.0     3.5  2005-04-02 23:53:47
1       1     29.0     3.5  2005-04-02 23:31:16
2       1     32.0     3.5  2005-04-02 23:33:39
3       1     47.0     3.5  2005-04-02 23:32:07
4       1     50.0     3.5  2005-04-02 23:29:40


DATA IMPUTATION

In [37]:
user_movie = dataset.pivot(
    index='userId',
    columns='movieId',
    values='rating'
)

user_movie = user_movie.fillna(0)

NORMALIZING THE RATING

In [38]:
user_movie[user_movie > 0] = 1

CONVERT TO NUMPY

In [39]:
training_data = user_movie.values
training_data = training_data.astype(np.float32)

BUILDING THE RBM

In [40]:
visible_neuron = training_data.shape[1]
hidden_neuron = 128

CREATING RBM ARCHITECTURE

In [41]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()

model.add(Dense(units=hidden_neuron, activation='relu', input_dim=visible_neuron))

model.add(Dense(units=visible_neuron, activation='sigmoid'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


COMPILING TEH MODEL

In [42]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam'
)

TRAINING MODEL

In [43]:
model.fit(
    training_data,
    training_data,
    batch_size=100,
    epochs=10
)

Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 15s 123ms/step - loss: 0.3332
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 13s 119ms/step - loss: 0.0735
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 13s 115ms/step - loss: 0.0508
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 21s 123ms/step - loss: 0.0389
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 20s 116ms/step - loss: 0.0338
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 13s 116ms/step - loss: 0.0300
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 14s 119ms/step - loss: 0.0276
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 14s 121ms/step - loss: 0.0257
Epoch 9/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 14s 122ms/step - loss: 0.0246
Epoch 10/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 20s 117ms/step - loss: 0.0238


MAKING RECOMMENDATION

PICKNG USER

In [44]:
user_id = 1
user_vector = training_data[user_id].reshape(1,-1)

PREDICT

In [45]:
model.predict(user_vector)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step


array([[1.64300018e-05, 8.96941274e-02, 1.17120715e-02, ...,
        2.11660426e-05, 1.86271991e-05, 3.14413737e-05]], dtype=float32)

RECOMMEND MOVIE

In [46]:
recommended_movies = np.argsort(model.predict(user_vector)[0])[::-1][:10]
print(recommended_movies)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
[ 257 1174 1161 2461  835 1178  582  586 1163  475]


ACCURACY SCORE

In [48]:
from sklearn.metrics import accuracy_score
predictions = (model.predict(user_vector)[0] > 0.5).astype(int)
accuracy_score(user_vector[0], predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step


0.9962275819418677